# Day 024：DPO 偏好数据与 DPO loss

对照 `minimind/dataset/lm_dataset.py::DPODataset` 和 `minimind/trainer/train_dpo.py`，
建立 chosen/rejected -> log probability -> relative preference -> DPO loss 的完整链路。
本 Notebook 只保留可复现实验和源码片段；没有伪造完整训练输出。

## 1. DPO 的位置

DPO 是 Direct Preference Optimization（直接偏好优化），属于大语言模型后训练中的离线偏好优化。
同一个 prompt 配一条 `chosen` 和一条 `rejected` 完整回答；训练时用冻结的 reference 作为锚点，
让可训练的 policy 相对于 reference 更偏向 chosen。它不是 PPO 式在线强化学习，但理论上由带 KL 约束的 RLHF 目标推导而来。

## 2. 数据和 next-token 对齐

`DPODataset` 对 chosen/rejected 分别应用同一个 chat template，`add_generation_prompt=False` 表示已有完整回答，
不是重新开启一次 assistant 生成。token 化并 padding 后，`x = ids[:-1]`、`y = ids[1:]`，mask 同步右移。
`generate_loss_mask()` 只标记 assistant 的 BOS/EOS 区间，prompt、角色标记和 padding 不参与回答分数。

In [ ]:
# 仅演示 next-token 对齐和 mask 的形状，不依赖模型权重
import torch

input_ids = torch.tensor([[10, 11, 12, 13, 14]])
loss_mask = torch.tensor([[0, 0, 1, 1, 1]])
x = input_ids[:, :-1]
y = input_ids[:, 1:]
mask = loss_mask[:, 1:]
print('x:', x.tolist())
print('y:', y.tolist())
print('mask:', mask.tolist())

预期现象：`x[t]` 对齐 `y[t]`，因此 mask 也要从第一个 label 位置开始。真实训练 batch 将 chosen 放在前半部分、rejected 放在后半部分；
例如各自 `[3, 10]`，拼接后 `x/y/mask` 都是 `[6, 10]`。

In [ ]:
import torch.nn.functional as F

def logits_to_log_probs(logits, labels):
    log_probs = F.log_softmax(logits, dim=2)
    return torch.gather(log_probs, 2, labels.unsqueeze(2)).squeeze(-1)

torch.manual_seed(0)
logits = torch.randn(2, 3, 5)
labels = torch.tensor([[1, 2, 0], [4, 1, 3]])
per_token = logits_to_log_probs(logits, labels)
print('logits shape:', tuple(logits.shape))
print('labels shape:', tuple(labels.shape))
print('per-token log_probs shape:', tuple(per_token.shape))
print('selected values:', per_token)

`logits_to_log_probs()` 只取每个位置真实目标 token 的 log probability：`[batch, seq, vocab] -> [batch, seq]`。
之后 `(per_token * mask).sum(dim=1)` 把每条完整回答压成一个 scalar。

In [ ]:
# 三组减法的数值含义
chosen_policy = torch.tensor([-2.0])
reject_policy = torch.tensor([-3.5])
chosen_ref = torch.tensor([-1.0])
reject_ref = torch.tensor([-1.5])
beta = 0.1
pi_logratios = chosen_policy - reject_policy
ref_logratios = chosen_ref - reject_ref
dpo_logits = pi_logratios - ref_logratios
loss = -F.logsigmoid(beta * dpo_logits)
print('pi_logratios:', pi_logratios.item())
print('ref_logratios:', ref_logratios.item())
print('dpo logits:', dpo_logits.item())
print('loss:', loss.item())

解释：

- `chosen_policy - rejected_policy = log(policy(chosen)/policy(rejected))`：policy 当前的相对偏好。
- `chosen_ref - rejected_ref`：reference 原本的相对偏好。
- 两者再相减：policy 相对于 reference 是否额外增强了 chosen 的偏好。

这些减法不会自己发现哪条回答客观更好；chosen/rejected 身份来自偏好数据。`-logsigmoid(beta * logits)` 将“chosen 应该胜出”变成可优化损失，
`beta` 控制偏离 reference 的尺度。真实 `train_epoch()` 还会把 `outputs.aux_loss` 加到 DPO loss 上（MoE 时）。

## 3. Day 024 验收

已确认：DPO 的定位和输入、chat template 与回答 mask、next-token 对齐、reference/policy 的梯度区别、
log probability 提取、chosen/rejected 配对、三个 log-ratio 减法、beta 和 preference loss，以及训练更新路径。
下一阶段恢复点：`minimind/trainer/train_grpo.py` 的 rollout 入口，先理解同一 prompt 生成多条回答和组内相对奖励。